# Quiz 2 - CIR Simulation and Caplet/Floorlet Pricing

This notebook is the main pipeline: set inputs once, run all cells, and get statistics, premiums, and plots.

In [24]:
import numpy as np
import pandas as pd

from src.cir_simulate import simulate_cir_paths, plot_cir_paths
from src.statistics_utils import terminal_rate_statistics
from src.caplet_pricing import (
    forward_looking_option_premium_bps,
    backward_looking_option_premium_bps,
)
from src.bond_option_fd import (
    check_explicit_cir_stability,
    explicit_cir_put_on_bond_premiums_bps,
)

pd.set_option('display.float_format', lambda x: f'{x:,.8f}')

In [ ]:
# Inputs to be provided during the quiz

# simulation settings
seed = 42  ################################ modify in quiz
n_paths = 1000 #1000
n_steps = 365 # number of days to simulate,
dt = 1 / 365 # time step in years 

# CIR parameters
kappa = 0.1 ################################ modify in quiz
theta = 0.05 ################################ modify in quiz
sigma = 0.08 ################################ modify in quiz
r0 = 0.04 ################################ modify in quiz

# cap/floor contract settings
cap_strike = 0.04 ################################ modify in quiz
reset_day = 275
settlement_day = 365

# question 4-5 (FD for put on 3-year ZCB)
put_strike = 0.9 ################################ modify in quiz
option_maturity = 1.0
bond_maturity = 3.0
fd_space_steps = 100
fd_dr = 0.001
fd_time_steps = 2000
fd_dt = 1.0 / fd_time_steps

# question 1-3

In [26]:
# Simulate and Statistics
times, rates = simulate_cir_paths(
    kappa=kappa,
    theta=theta,
    sigma=sigma,
    r0=r0,
    n_paths=n_paths,
    n_steps=n_steps,
    dt=dt,
    seed=seed,
)

stats = terminal_rate_statistics(rates)
print('Terminal short-rate descriptive statistics (day 365):')
print(np.shape(rates))
print(stats)
# plot_cir_paths(rates, times, reset_day=reset_day, settlement_day=settlement_day)

Terminal short-rate descriptive statistics (day 365):
(100000, 366)
mean        0.04097674
std         0.01535672
q1          0.02986179
q2_median   0.03950878
q3          0.05046196
dtype: float64


In [27]:
# Caplet Premiums in bps
fwd_caplet = forward_looking_option_premium_bps(
    rates=rates,
    strike=cap_strike,
    reset_index=reset_day,
    settlement_index=settlement_day,
    dt=dt,
    is_cap=True,
)
bwd_caplet = backward_looking_option_premium_bps(
    rates=rates,
    strike=cap_strike,
    reset_index=reset_day,
    settlement_index=settlement_day,
    dt=dt,
    is_cap=True,
)

print( "fwd_caplet premium (in bps):")
print(fwd_caplet)
print( "bwd_caplet premium (in bps):")
print(bwd_caplet)

fwd_caplet premium (in bps):
13.292823139631908
bwd_caplet premium (in bps):
14.0180940625743


# question 4,5

In [28]:
# Explicit FD grid stability check
import importlib
import src.bond_option_fd as bond_option_fd

importlib.reload(bond_option_fd)
r_max = fd_space_steps * fd_dr
is_stable, stability_msg = bond_option_fd.check_explicit_cir_stability(
    kappa=kappa,
    theta=theta,
    sigma=sigma,
    r_max=r_max,
    dr=fd_dr,
    dt=fd_dt,
)

print('Explicit FD grid check:', stability_msg)
if not is_stable:
    raise ValueError('Grid is unstable. Adjust dr/dt before pricing.')

Explicit FD grid check: v1=500.000000, v2=0.500000, A in [0.000375, 0.159625], B in [0.683150, 0.996800], C in [0.002825, 0.157175], stable=True


In [29]:
# Q4-5: European and American put premiums (bps) on 3Y ZCB, option maturity 1Y
import importlib
import src.bond_option_fd as bond_option_fd

importlib.reload(bond_option_fd)
fd_result = bond_option_fd.explicit_cir_put_on_bond_premiums_bps(
    kappa=kappa,
    theta=theta,
    sigma=sigma,
    r0=r0,
    option_maturity=option_maturity,
    bond_maturity=bond_maturity,
    strike=put_strike,
    n_r_steps=fd_space_steps,
    dr=fd_dr,
    n_t_steps=fd_time_steps,
    dt=fd_dt,
)

print(f"European put premium (bps): {fd_result['european_put_bps']:.6f}")
print(f"American put premium (bps): {fd_result['american_put_bps']:.6f}")

European put premium (bps): 32.356621
American put premium (bps): 158.676012


# not for quiz

In [30]:
# Theoretical CIR terminal moments check
T = n_steps * dt
exp_term = np.exp(-kappa * T)

theoretical_mean = theta + (r0 - theta) * exp_term
theoretical_variance = (
    r0 * (sigma**2 / kappa) * (exp_term - np.exp(-2 * kappa * T))
    + theta * (sigma**2 / (2 * kappa)) * (1 - exp_term) ** 2
)

mc_mean = rates[:, -1].mean()
mc_variance = rates[:, -1].var(ddof=1)

print('CIR terminal moment check at T =', T)
print(f'Theoretical E[r_T]:   {theoretical_mean:.8f}')
print(f'Monte Carlo E[r_T]:   {mc_mean:.8f}')
print(f'Theoretical Var[r_T]: {theoretical_variance:.8f}')
print(f'Monte Carlo Var[r_T]: {mc_variance:.8f}')

CIR terminal moment check at T = 1.0
Theoretical E[r_T]:   0.04095163
Monte Carlo E[r_T]:   0.04097674
Theoretical Var[r_T]: 0.00023492
Monte Carlo Var[r_T]: 0.00023583


In [31]:
from scipy.stats import ncx2

# Simple CIR closed-form benchmark for forward-looking caplet (via ZCB put)
t, s = reset_day * dt, settlement_day * dt
tau = s - t
h = np.sqrt(kappa**2 + 2*sigma**2)
B0t = 2*(np.exp(h*t)-1)/(2*h + (kappa+h)*(np.exp(h*t)-1)); B0s = 2*(np.exp(h*s)-1)/(2*h + (kappa+h)*(np.exp(h*s)-1)); Bts = 2*(np.exp(h*(s-t))-1)/(2*h + (kappa+h)*(np.exp(h*(s-t))-1))
A0t = ((2*h*np.exp((kappa+h)*t/2))/(2*h + (kappa+h)*(np.exp(h*t)-1)))**(2*kappa*theta/sigma**2); A0s = ((2*h*np.exp((kappa+h)*s/2))/(2*h + (kappa+h)*(np.exp(h*s)-1)))**(2*kappa*theta/sigma**2); Ats = ((2*h*np.exp((kappa+h)*(s-t)/2))/(2*h + (kappa+h)*(np.exp(h*(s-t))-1)))**(2*kappa*theta/sigma**2)
P0t = A0t*np.exp(-B0t*r0); P0s = A0s*np.exp(-B0s*r0)
X = 1/(1 + cap_strike*tau)  # bond-strike equivalent
rho = 2*h/(sigma**2*(np.exp(h*t)-1)); psi = (kappa+h)/sigma**2; df = 4*kappa*theta/sigma**2
ncps = 2*rho**2*r0*np.exp(h*t)/(rho+psi+Bts); ncpt = 2*rho**2*r0*np.exp(h*t)/(rho+psi)
z = np.log(Ats/X)/Bts
call_bond = P0s*ncx2.cdf(2*z*(rho+psi+Bts), df, ncps) - X*P0t*ncx2.cdf(2*z*(rho+psi), df, ncpt)
put_bond = call_bond - P0s + X*P0t
caplet_cf_bps = (1 + cap_strike*tau)*put_bond*10000

print(f"CIR closed-form forward caplet (bps): {caplet_cf_bps:.6f}")
print(f"MC forward caplet (current implementation, bps): {fwd_caplet:.6f}")

CIR closed-form forward caplet (bps): 13.685831
MC forward caplet (current implementation, bps): 13.292823


In [32]:
# Closed-form CIR European put on ZCB using root/src implementation
import sys
from pathlib import Path

repo_src = Path.cwd().parent / "src"  # root/src outside quiz2
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

from fi_pricing.models.affine import CIRModel

cir_model = CIRModel(kappa=kappa, theta=theta, sigma=sigma)
put_closed = cir_model.zcb_option(
    t=0.0,
    T=bond_maturity,
    rt=r0,
    T_expiry=option_maturity,
    K=put_strike,
    option_type="put",
)
put_closed_bps = 10000 * float(put_closed)

print(f"Closed-form European put (bps) [root/src]: {put_closed_bps:.6f}")
print(f"FD European put (bps):                     {fd_result['european_put_bps']:.6f}")
print(f"Difference (FD - closed, bps):             {fd_result['european_put_bps'] - put_closed_bps:.6f}")
print("American put: no simple CIR closed-form; FD/tree methods are standard.")

Closed-form European put (bps) [root/src]: 32.630769
FD European put (bps):                     32.356621
Difference (FD - closed, bps):             -0.274148
American put: no simple CIR closed-form; FD/tree methods are standard.
